# INF 791 - Tópicos Especiais II – Redes Complexas
## Projeto Final: Análise Estrutural de Sentimentos sobre Entidades em Eventos utilizando Redes Bipartidas com Sinais
**Discente**: Pedro Henrique Silva Oliveira (EF02677)  
**Docente**: Julio Cesar Soares dos Reis  
**Semestre**: 1º Semestre de 2026  
**Repositório**: [GitHub](https://github.com/pedrohso7/tf-redes-complexas)


## 1. Introdução e Motivação

O gerenciamento de eventos acadêmicos e corporativos modernos tem se apoiado em soluções móveis e ubíquas como a plataforma **myMobiConf** (Silva et al., 2024; Oliveira et al., 2024). Durante um evento, os participantes enviam feedbacks textuais voluntários sobre diversos aspectos (entidades) como palestrantes, sessões, internet Wi-Fi ou coffee break. Embora o sistema preserve a privacidade (dados descaracterizados), é possível correlacionar as diferentes manifestações enviadas pelo mesmo usuário anônimo ao longo do evento através de seu identificador de sessão.

Tradicionalmente, a análise de feedback de eventos na indústria baseia-se em métricas agregadas agregadas, como o **Net Sentiment Score (NSS)**:

$$\text{NSS} = \frac{\text{Comentários Positivos} - \text{Comentários Negativos}}{\text{Total de Comentários}} \times 100$$

No entanto, o NSS simplifica e destrói a topologia da rede de opiniões de cada evento ao agregar os dados de forma puramente percentual. Ela falha em responder perguntas críticas sobre a estrutura das relações:
1. **Heterogeneidade de Usuários**: A insatisfação partiu de vários participantes independentes que comentaram uma única vez (crítica pulverizada) ou de um único usuário altamente engajado que enviou múltiplos feedbacks (crítica centralizada)?
2. **Bolhas de Opinião**: Os usuários que expressam sentimentos semelhantes sobre as mesmas entidades formam comunidades estruturais de percepção comum?
3. **Associação e Correlação de Aspectos**: A avaliação negativa de uma entidade (ex: Wi-Fi) está estatisticamente vinculada à avaliação de outra entidade (ex: Organização), indicando uma contaminação da percepção da experiência global?

### Hipóteses de Pesquisa
Este trabalho parte da hipótese de que **a percepção de sentimentos de um evento possui uma estrutura de rede bipartite sinalizada cuja topologia revela padrões de heterogeneidade de usuários, bolhas de opinião de participantes e correlação de aspectos do evento que não podem ser detectados pelas métricas estatísticas tradicionais**.

### Questões de Pesquisa (QP)
O projeto é orientado por cinco Questões de Pesquisa centrais aplicadas ao conjunto de dados de cada evento:
* **QP1 (Concentração e Polarização)**: Quais entidades de um evento concentram a maior carga de sentimentos positivos, negativos ou neutros, e qual é o saldo estrutural de sentimento de cada uma?
* **QP2 (Heterogeneidade dos Usuários)**: Como se distribui o volume de feedback por usuário? A atividade de feedback é pulverizada de forma homogênea ou dominada por poucos super-usuários engajados?
* **QP3 (Bolhas de Opinião Compartilhada)**: É possível agrupar os usuários em comunidades baseando-se unicamente na similaridade de suas percepções sobre as mesmas entidades (projeção unipartida de usuários)?
* **QP4 (Correlação Estrutural de Aspectos)**: Quais entidades ou serviços do evento tendem a ser avaliados conjuntamente pelos mesmos participantes, revelando padrões de dependência de experiência (projeção unipartida de entidades)?
* **QP5 (Sentimento Geral do Evento)**: Como o sentimento geral e agregado do evento (NSS Global tradicional de mercado) se comporta e se compara com as métricas estruturais do grafo (como o peso médio das arestas e as comunidades da rede bipartida sinalizada)?


## 2. Trabalhos Relacionados

A modelagem estrutural de sentimentos utilizando a teoria de redes complexas é uma abordagem moderna. O artigo base de **Nonaka e Perry (2026)**, *"Evaluating LLM Story Generation through Large-scale Network Analysis of Social Structures"*, propôs analisar a estrutura narrativa de histórias geradas por IA modelando-as como redes unipartidas sinalizadas de personagens (*signed character networks*). Suas métricas avaliam o viés de modelos de linguagem (LLMs) em criar histórias excessivamente lineares e amigáveis, demonstrando a utilidade analítica das redes sinalizadas para caracterizar tendências qualitativas de dados de linguagem natural.

Enquanto Nonaka e Perry (2026) trabalham com grafos unipartidos de interações de personagens fictícios, este projeto estende o conceito para **Redes Bipartidas com Sinais** formadas por feedbacks de eventos do mundo físico. A natureza do myMobiConf exige mapear as interações assimétricas entre **Usuários** (que comentam) e **Entidades** (palestrantes, infraestrutura, organização), aplicando projeções unipartidas sinalizadas em ambos os conjuntos para identificar bolhas de opinião de participantes (projeção de usuários) e agrupamento de dependência de aspectos (projeção de entidades).


## 3. Metodologia

A modelagem baseia-se na extração de entidades e sentimentos dos logs de comentários salvos de cada evento, seguindo um fluxo metodológico estruturado.

### 3.1. Workflow Metodológico Geral

O pipeline metodológico que guia a pesquisa e o processamento de dados deste projeto é composto por sete etapas sequenciais estruturadas:

1. **Extração de comentários**: Carga e caracterização descritiva dos dados reais extraídos do sistema *myMobiConf* para os Eventos 1 a 6, definindo a base da análise.
2. **Processamento e Filtro de Ruído Estrutural**: Pré-processamento e descarte de feedbacks globais genéricos que não mencionam aspectos específicos.
3. **Extração de Aspectos**: Identificação e extração de entidades específicas (ex: Palestrantes, Wi-Fi, Coffee Break) mencionadas nos feedbacks.
4. **Análise de Sentimento**: Classificação do sentimento associado a cada aspecto e cálculo de intensidade contínua no intervalo $[-1, 1]$.
5. **Modelagem de Rede Bipartida Sinalizada**: Construção do grafo bipartido direcionado com pesos de arestas representados pelas intensidades de sentimentos.
6. **Projeções de Redes Unipartidas**: Geração das redes secundárias de relacionamento de usuários ($G_{user}$) e de associação de entidades ($G_{entity}$).
7. **Análise Topológica e Comparação de Métricas**: Execução de algoritmos de comunidades (Louvain) e cálculo de métricas estruturais vs. NSS agregado.

### 3.2. Extração de comentários

A fim de validar a metodologia proposta, foram utilizados dados reais extraídos da plataforma de suporte a eventos *myMobiConf*. Os comentários foram coletados de forma totalmente anonimizada, preservando a privacidade dos participantes por meio da substituição de seus identificadores por hashes alfanuméricos únicos (UUID).

Para garantir que o método de redes complexas seja robusto diante de diferentes perfis e volumes de engajamento, os conjuntos de dados (doravante denominados *Eventos 1 a 6*) abrangem eventos com dinâmicas de feedback variadas. Apresenta-se abaixo a distribuição do volume de comentários (indicador de engajamento de feedback) para cada evento em ordem decrescente:

| Evento | Número de comentários | Engajamento |
| :--- | :---: | :---: |
| **Evento 1** | 1.858 | Muito Alto |
| **Evento 2** | 1.207 | Muito Alto |
| **Evento 3** | 643 | Alto |
| **Evento 4** | 76 | Moderado |
| **Evento 5** | 25 | Baixo |
| **Evento 6** | 13 | Micro |

#### Racional para Análises Separadas por Grafo
Serão realizadas análises estruturais e de sentimentos separadas para cada grafo (evento) a fim de alcançar os objetivos definidos na pesquisa. Essa segmentação por grafo é essencial pelas seguintes justificativas metodológicas:
1. **Independência de Contexto**: Cada evento possui entidades próprias (palestrantes, locais, infraestrutura) e dinâmicas de público específicas. Misturar os dados destruiria a integridade topológica do feedback de cada conferência.
2. **Validação de Perfis de Engajamento (QP2 e QP5)**: Avaliar eventos com volumes de engajamento contrastantes (desde 13 até 1.858 feedbacks) permite analisar como o engajamento individual (comentários por usuário) se distribui. Um evento com menos participantes mas de alta densidade de participação pode produzir uma topologia bipartida mais coesa e dinâmica do que um evento massivo com baixa taxa de participação individual, afetando diretamente a diluição do sentimento geral (QP5).
3. **Modularidade das Bolhas (QP3)**: Permite observar se o número e a coesão das bolhas de opinião (comunidades de concordância na rede de usuários) escalam de forma linear ou sublinear com a quantidade de participantes ativos e seu respectivo grau de envolvimento.
4. **Padrões de Co-ocorrência (QP4)**: Identificar se o acoplamento de aspectos (ex: infraestrutura vs. palestras na rede de entidades) é uma característica universal de eventos gerenciados pela plataforma ou se é específico de determinadas escalas de público.

#### 3.2.1. Limpeza e Filtro de Ruído Léxico

Esta etapa realiza a primeira fase de tratamento e saneamento estrutural de dados textuais de comentários brutos. O objetivo é remover ruídos de integridade e estabilizar o código antes de qualquer modelagem semântica ou extração de aspectos. Serão aplicados os seguintes critérios:
1. **Tratamento de Registros Vazios ou Nulos (NaN)**: Descarte de feedbacks nulos sem conteúdo textual (necessário para evitar erros de tipo no Python durante o processamento de NLP).
2. **Deduplicação de Logs**: Remoção de logs duplicados contendo exatamente o mesmo texto enviado pelo mesmo participante, evitando inflar artificialmente o grau dos nós ou o peso ponderado dos sentimentos.
3. **Padronização Textual Básica**: Remoção de espaços nas pontas e normalização para caixa baixa (`lower`).

**Nota Importante de Projeto**: Qualquer outra filtragem de ruído semântico (como saudações como 'oi', pontuações repetidas como '...' ou erros de digitação e typos) é deliberadamente **delegada à etapa seguinte de mapeamento de aspectos (Etapa 2) via Fuzzy Matching**. Isso ocorre porque qualquer feedback que não contenha uma menção aproximada ou exata a um aspecto de interesse será naturalmente mapeado como `None` e não gerará arestas na rede, mantendo a simplicidade e robustez matemática do pipeline.

Abaixo, apresenta-se a tabela comparativa do volume de comentários brutos originais contra os comentários filtrados e deduplicados nesta etapa:

| Evento | Comentários Brutos | Comentários Filtrados (Etapa 1) |
| :--- | :---: | :---: |
| **Evento 1** | 1.857 | 1.796 |
| **Evento 2** | 1.206 | 1.198 |
| **Evento 3** | 642 | 599 |
| **Evento 4** | 75 | 74 |
| **Evento 5** | 24 | 24 |
| **Evento 6** | 12 | 12 |


In [ ]:
# Implementação da Limpeza e Extração Inicial (Etapa 1)
import os
import pandas as pd

def processar_limpeza_inicial(nome_arquivo):
    pasta_origem = "comentarios_extraidos"
    pasta_destino = "extracao1"
    os.makedirs(pasta_destino, exist_ok=True)
    
    caminho_origem = os.path.join(pasta_origem, nome_arquivo)
    if not os.path.exists(caminho_origem):
        print(f"Erro: O arquivo '{caminho_origem}' não foi encontrado.")
        return None
    
    # Leitura do CSV
    df = pd.read_csv(caminho_origem)
    total_inicial = len(df)
    
    # 1. Remover nulos na coluna comentário
    df = df.dropna(subset=['comentário']).copy()
    
    # 2. Deduplicação (comentários idênticos pelo mesmo participante)
    df = df.drop_duplicates(subset=['participante', 'comentário']).copy()
    
    # 3. Normalização básica
    df['comentário'] = df['comentário'].astype(str).str.strip().str.lower()
    
    # Salvar resultado
    nome_sem_ext, ext = os.path.splitext(nome_arquivo)
    nome_saida = f"{nome_sem_ext}_extracao1{ext}"
    caminho_destino = os.path.join(pasta_destino, nome_saida)
    df.to_csv(caminho_destino, index=False)
    
    print(f"Limpeza concluída para {nome_arquivo}!")
    print(f" - Comentários originais: {total_inicial}")
    print(f" - Comentários após Etapa 1: {len(df)}")
    print(f" - Salvo em: {caminho_destino}")
    return caminho_destino

# Exemplo de execução no notebook (comentado por padrão):
# processar_limpeza_inicial("comentarios_wit.csv")


### 3.3. Processamento e Filtro de Ruído Estrutural

Feedbacks puramente genéricos sobre o evento que não mencionam entidades específicas (ex.: *"O evento foi muito legal!"* ou *"Gostei muito de hoje"*) são descartados para manter a integridade da rede bipartida orientada a aspectos. Esse descarte é crucial porque comentários puramente gerais não vinculam um usuário a uma entidade do evento, representando ruído que inflaria artificialmente a centralidade global ou criaria caminhos espúrios nas projeções.

### 3.4. Extração de Aspectos

Identificação automática de aspectos e entidades físicas ou organizacionais citadas no corpo de texto (ex.: Palestrante A, Palestrante B, Internet Wi-Fi, Coffee Break, Ar Condicionado, Organização). Cada comentário filtrado é mapeado para uma das entidades mapeadas.

### 3.5. Análise de Sentimento

Processamento de linguagem natural (NLP) para classificar a manifestação associada a cada aspect. Em vez de uma classificação discreta simples (positivo, negativo, neutro), estima-se uma intensidade contínua de sentimento no intervalo $[-1, 1]$ com base nas probabilidades ou logites do modelo de linguagem (por exemplo, modelos pré-treinados tipo BERT ou RoBERTa ajustados para análise de sentimentos).

### 3.6. Modelagem da Rede Bipartida Sinalizada

A rede principal é modelada como um Grafo Bipartido Sinalizado e Direcionado $G_{bipartido} = (U, E, A, w)$:
- **Nós**: Compostos por dois conjuntos disjuntos de nós: Usuários ($U$, representando os participantes) e Entidades ($E$, representando os aspectos do evento).
- **Arestas**: As arestas direcionadas $A \subseteq U \times E$ representam o ato de um usuário expressar uma opinião sobre uma entidade.
- **Pesos**: A função de peso $w: A \rightarrow [-1, 1]$ indica a intensidade contínua do sentimento correspondente à avaliação do usuário sobre aquela entidade. Se um usuário comenta múltiplas vezes sobre uma mesma entidade, a aresta final recebe a média dos scores correspondentes.

### 3.7. Projeções de Redes Unipartidas

Para responder às questões de pesquisa focadas em usuários (QP3) e em aspectos (QP4), deduzimos duas projeções unipartidas sinalizadas a partir do grafo bipartido:

1. **Projeção de Usuários ($G_{user} = (U, A_{user}, w_{user})$)**:
   - Grafo unipartido sinalizado onde as arestas conectam usuários $u_1, u_2 \in U$ se co-comentaram as mesmas entidades.
   - O peso $w_{user}(u_1, u_2)$ representa a concordância média de suas opiniões sobre essas entidades:
     $$w_{user}(u_1, u_2) = \frac{1}{|E_{comum}|} \sum_{e \in E_{comum}} w(u_1, e) \cdot w(u_2, e)$$
     Se ambos elogiam ou ambos criticam as mesmas entidades, a aresta é positiva. Se divergem, é negativa.

2. **Projeção de Entidades ($G_{entity} = (E, A_{entity}, w_{entity})$)**:
   - Grafo unipartido sinalizado onde as arestas conectam entidades $e_1, e_2 \in E$ se foram comentadas pelos mesmos usuários.
   - O peso $w_{entity}(e_1, e_2)$ representa o saldo de co-avaliação e correlação estrutural dos aspectos:
     $$w_{entity}(e_1, e_2) = \left( \frac{1}{|U_{comum}|} \sum_{u \in U_{comum}} w(u, e_1) \cdot w(u, e_2) \right) \cdot |U_{comum}|$$
     Avalia se os participantes tendem a ter a mesma opinião sobre ambos os aspectos (peso positivo) ou se a avaliação de um correlaciona com a do outro de forma inversa (peso negativo), ponderada pelo volume de avaliadores comuns.

### 3.8. Análise Topológica e Comparação de Métricas

Nesta etapa, extraímos as métricas da rede estrutural para responder às QPs e confrontar com o Net Sentiment Score (NSS) clássico de mercado:
* **Grau e Grau Ponderado**: Mede o volume de feedbacks por usuário (Out-Degree) e a popularidade ou sentimento líquido das entidades (Weighted In-Degree).
* **Densidade Bipartida**: Fração de pares de feedbacks realizados em relação ao total possível.
* **Detecção de Comunidades nas Projeções**: Segmentação de bolhas de usuários com o mesmo perfil de percepção ($G_{user}$) e agrupamento de entidades associadas na experiência ($G_{entity}$).
* **Comparação NSS vs. Métricas do Grafo**: Comparação empírica entre o NSS clássico, o NSS focado em aspectos e o peso médio das arestas do grafo bipartido.


In [ ]:
# 1. Configurações e Importações de Bibliotecas
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from networkx.algorithms import bipartite
from networkx.algorithms import community
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'


In [ ]:
# 2. Simulação de Dados de Comentários de um Evento (myMobiConf)
def gerar_dados_evento_simulado(semente=42, usar_dados_esparsos=False):
    np.random.seed(semente)
    entidades = ["Palestrante A", "Palestrante B", "Internet Wi-Fi", "Coffee Break", "Organização", "Ar Condicionado"]
    usuarios = [f"Usuario_{i:02d}" for i in range(1, 26)]
    
    # Comentários baseados em aspectos
    comentarios_exemplos = {
        "positivo": [
            "A palestra do Palestrante A foi incrível, muito didática!",
            "O coffee break estava excelente, muitas opções gostosas.",
            "Organização impecável do evento hoje.",
            "Palestrante B trouxe insights muito inovadores para a área."
        ],
        "negativo": [
            "A internet Wi-Fi está caindo toda hora, impossível usar.",
            "O ar condicionado da sala principal estava congelante e barulhento.",
            "O coffee break acabou rápido demais, fila enorme.",
            "Palestrante A estourou o tempo e não abriu para perguntas."
        ],
        "neutro": [
            "O Palestrante B apresentou conceitos básicos que eu já conhecia.",
            "A organização mudou a sala de última hora.",
            "Ar condicionado estava desligado, mas a sala estava fresca."
        ],
        "geral": [
            "O evento está muito bom de forma geral!",
            "Gostando bastante do dia de hoje.",
            "Que dia produtivo!"
        ]
    }
    
    dados = []
    for u in usuarios:
        # Número de comentários por usuário (simula variação de engajamento)
        n_coments = np.random.randint(1, 3) if usar_dados_esparsos else np.random.randint(1, 5)
        for _ in range(n_coments):
            tipo = np.random.choice(["positivo", "negativo", "neutro", "geral"], p=[0.40, 0.40, 0.15, 0.05])
            texto = np.random.choice(comentarios_exemplos[tipo])
            
            # Extração de aspectos
            if tipo == "geral":
                entidade = None
            else:
                if "Palestrante A" in texto:
                    entidade = "Palestrante A"
                elif "Palestrante B" in texto:
                    entidade = "Palestrante B"
                elif "Wi-Fi" in texto or "internet" in texto:
                    entidade = "Internet Wi-Fi"
                elif "coffee" in texto:
                    entidade = "Coffee Break"
                elif "ar condicionado" in texto or "sala" in texto:
                    entidade = "Ar Condicionado"
                else:
                    entidade = "Organização"
            
            # Intensidade do sentimento contínuo extraída pelo NLP
            if tipo == "positivo":
                score = np.random.uniform(0.3, 1.0)
            elif tipo == "negativo":
                score = np.random.uniform(-1.0, -0.3)
            elif tipo == "neutro":
                score = np.random.uniform(-0.29, 0.29)
            else:
                score = np.random.uniform(0.1, 0.5)
                
            dados.append({
                "usuario": u,
                "texto": texto,
                "entidade_mencionada": entidade,
                "score_sentimento": score
            })
            
    df_comentarios = pd.DataFrame(dados)
    return df_comentarios

df_comentarios = gerar_dados_evento_simulado(semente=42)
print(f"Total de comentários coletados no evento: {len(df_comentarios)}")
print(df_comentarios.head())


In [ ]:
# 3. Pipeline de Processamento ABSA e Filtro de Ruído
def processar_pipeline_absa(df):
    print("--- Executando Pipeline de NLP e Filtro de Ruído ---")
    
    # Filtro de ruído: descarta feedbacks globais sem entidades específicas
    df_filtrado = df[df['entidade_mencionada'].notna()].copy()
    print(f"Comentários totais extraídos: {len(df)}")
    print(f"Comentários válidos (mencionando entidades): {len(df_filtrado)}")
    print(f"Removidos {len(df) - len(df_filtrado)} comentários gerais (ruído estrutural).")
    
    # Categorização descritiva para fins estatísticos
    def categorizar(score):
        if score > 0.3: return 'Positivo'
        elif score < -0.3: return 'Negativo'
        else: return 'Neutro'
        
    df_filtrado['categoria_sentimento'] = df_filtrado['score_sentimento'].apply(categorizar)
    return df_filtrado

df_processado = processar_pipeline_absa(df_comentarios)
print(df_processado.head())


In [ ]:
# 4. Construção da Rede Bipartida com Sinais
def construir_rede_bipartida(df):
    G = nx.DiGraph()
    
    usuarios = df['usuario'].unique().tolist()
    entidades = df['entidade_mencionada'].unique().tolist()
    
    # Atribuir o grupo bipartido aos nós
    G.add_nodes_from(usuarios, bipartite=0, label='Usuario')
    G.add_nodes_from(entidades, bipartite=1, label='Entidade')
    
    # Agrupar por par Usuário-Entidade tirando a média dos sentimentos
    edges_agg = df.groupby(['usuario', 'entidade_mencionada'])['score_sentimento'].mean().reset_index()
    
    for _, row in edges_agg.iterrows():
        G.add_edge(row['usuario'], row['entidade_mencionada'], weight=row['score_sentimento'])
        
    return G

G_bipartida = construir_rede_bipartida(df_processado)
n_usuarios = len([n for n, d in G_bipartida.nodes(data=True) if d.get('bipartite')==0])
n_entidades = len([n for n, d in G_bipartida.nodes(data=True) if d.get('bipartite')==1])
print(f"Grafo Bipartido Sinalizado Construído:")
print(f" - Nós de Usuários (U): {n_usuarios}")
print(f" - Nós de Entidades (E): {n_entidades}")
print(f" - Conexões de feedback sinalizadas: {G_bipartida.number_of_edges()}")


In [ ]:
# 5. Construção das Projeções Unipartidas (Usuários e Entidades)
def construir_projeção_usuarios(G_bip):
    """
    Projeta a rede de relacionamento de usuários.
    Conecta usuários que avaliaram as mesmas entidades. O peso representa a concordância das opiniões.
    """
    G_user = nx.Graph()
    usuarios = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 0]
    G_user.add_nodes_from(usuarios)
    
    for i in range(len(usuarios)):
        for j in range(i + 1, len(usuarios)):
            u1 = usuarios[i]
            u2 = usuarios[j]
            
            entidades_comuns = set(G_bip.successors(u1)).intersection(set(G_bip.successors(u2)))
            
            if entidades_comuns:
                produtos = []
                for ent in entidades_comuns:
                    w1 = G_bip[u1][ent]['weight']
                    w2 = G_bip[u2][ent]['weight']
                    produtos.append(w1 * w2)
                
                peso_final = np.mean(produtos)
                G_user.add_edge(u1, u2, weight=peso_final, co_ocorrencias=len(entidades_comuns))
    return G_user

def construir_projeção_entidades(G_bip):
    """
    Projeta a rede de relacionamento de entidades (aspectos).
    Conecta entidades se foram co-comentadas pelo mesmo usuário. O peso representa a co-avaliação estrutural.
    """
    G_ent = nx.Graph()
    entidades = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 1]
    G_ent.add_nodes_from(entidades)
    
    for i in range(len(entidades)):
        for j in range(i + 1, len(entidades)):
            e1 = entidades[i]
            e2 = entidades[j]
            
            usuarios_comuns = set(G_bip.predecessors(e1)).intersection(set(G_bip.predecessors(e2)))
            
            if usuarios_comuns:
                produtos = []
                for u in usuarios_comuns:
                    w1 = G_bip[u][e1]['weight']
                    w2 = G_bip[u][e2]['weight']
                    produtos.append(w1 * w2)
                
                # Pondera a similaridade média pelo volume de avaliadores em comum
                peso_final = np.mean(produtos) * len(usuarios_comuns)
                G_ent.add_edge(e1, e2, weight=peso_final, co_ocorrencias=len(usuarios_comuns))
    return G_ent

G_usuarios = construir_projeção_usuarios(G_bipartida)
G_entidades_proj = construir_projeção_entidades(G_bipartida)
print(f"Projeções Construídas:")
print(f" - Rede de Usuários: {G_usuarios.number_of_nodes()} nós, {G_usuarios.number_of_edges()} arestas")
print(f" - Rede de Entidades: {G_entidades_proj.number_of_nodes()} nós, {G_entidades_proj.number_of_edges()} arestas")


In [ ]:
# 6. Análise de Métricas Estruturais (QP1 e QP2)
def calcular_metricas(G_bip, G_user, G_ent_proj):
    entidades = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 1]
    usuarios = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 0]
    
    # 6.1. Análise de Entidades (QP1 - Concentração e Polarização)
    dados_entidades = []
    for ent in entidades:
        in_edges = G_bip.in_edges(ent, data=True)
        in_degree = len(in_edges)
        pesos = [d['weight'] for _, _, d in in_edges]
        
        in_degree_ponderado = sum(pesos)
        nss_estrutural = (in_degree_ponderado / in_degree * 100) if in_degree > 0 else 0
        
        pos_count = sum(1 for w in pesos if w > 0.3)
        neg_count = sum(1 for w in pesos if w < -0.3)
        neu_count = sum(1 for w in pesos if -0.3 <= w <= 0.3)
        
        dados_entidades.append({
            'Entidade': ent,
            'Comentários': in_degree,
            'Grau Ponderado': round(in_degree_ponderado, 2),
            'NSS Estrutural (%)': round(nss_estrutural, 2),
            'Positivos': pos_count,
            'Negativos': neg_count,
            'Neutros': neu_count
        })
        
    df_ent = pd.DataFrame(dados_entidades).sort_values(by='Comentários', ascending=False)
    
    # 6.2. Análise de Usuários (QP2 - Heterogeneidade e Engajamento)
    out_degrees = dict(G_bip.out_degree())
    df_usr = pd.DataFrame({
        'Usuario': usuarios,
        'Feedbacks Enviados': [out_degrees.get(u, 0) for u in usuarios]
    }).sort_values(by='Feedbacks Enviados', ascending=False)
    
    # Métricas globais bipartidas
    dens_bip = bipartite.density(G_bip, usuarios)
    print(f"Densidade Bipartida: {dens_bip:.4f}")
    print(f"Grau Médio de Feedback dos Usuários: {df_usr['Feedbacks Enviados'].mean():.2f}")
    
    # 6.3. Sentimento Geral do Evento (QP5 - NSS Global vs Peso Médio da Rede)
    # NSS Tradicional (inclui todos os comentários, inclusive ruídos gerais)
    total_comentarios = len(df_comentarios)
    pos_globais = sum(1 for w in df_comentarios['score_sentimento'] if w > 0.3)
    neg_globais = sum(1 for w in df_comentarios['score_sentimento'] if w < -0.3)
    nss_global_tradicional = ((pos_globais - neg_globais) / total_comentarios * 100) if total_comentarios > 0 else 0
    
    # NSS apenas dos aspectos filtrados
    total_filtrados = len(df_processado)
    pos_filt = sum(1 for w in df_processado['score_sentimento'] if w > 0.3)
    neg_filt = sum(1 for w in df_processado['score_sentimento'] if w < -0.3)
    nss_global_aspectos = ((pos_filt - neg_filt) / total_filtrados * 100) if total_filtrados > 0 else 0
    
    # Peso médio das arestas (sentimento contínuo médio no grafo bipartido)
    pesos_rede = [d['weight'] for _, _, d in G_bip.edges(data=True)]
    peso_medio_rede = np.mean(pesos_rede) if pesos_rede else 0.0
    
    print("\n--- Análise Comparativa do Sentimento Geral do Evento (QP5) ---")
    print(f"NSS Global Tradicional (Métrica de Mercado): {nss_global_tradicional:.2f}%")
    print(f"NSS Global Filtrado (Foco em Aspectos/Entidades): {nss_global_aspectos:.2f}%")
    print(f"Peso Médio das Arestas do Grafo Bipartido (Intensidade): {peso_medio_rede:.3f} (Escala [-1, +1])")
    
    return df_ent, df_usr

df_entidades, df_usuarios_atividade = calcular_metricas(G_bipartida, G_usuarios, G_entidades_proj)
print("\n--- Tabela de Análise das Entidades (QP1) ---")
print(df_entidades.to_string(index=False))
print("\n--- Distribuição de Atividade dos Usuários (QP2) ---")
print(df_usuarios_atividade.head(10).to_string(index=False))


In [ ]:
# 7. Detecção de Bolhas de Percepção e Agrupamento de Aspectos (QP3 e QP4)
def detectar_comunidades_projeções(G_user, G_ent_proj):
    # 7.1. Comunidades de Usuários (QP3) - Louvain
    # Filtra apenas arestas positivas (concordância) para agrupar usuários que pensam parecido
    G_user_positive = nx.Graph()
    G_user_positive.add_nodes_from(G_user.nodes())
    for u, v, d in G_user.edges(data=True):
        if d['weight'] > 0: # concordância positiva
            G_user_positive.add_edge(u, v, weight=d['weight'])
            
    comunidades_usr = community.louvain_communities(G_user_positive, weight='weight', seed=42)
    partition_usr = {}
    for i, comm in enumerate(comunidades_usr):
        for node in comm:
            partition_usr[node] = i
            
    print(f"Total de {len(comunidades_usr)} bolhas de opinião (comunidades de usuários) identificadas.")
    for i, comm in enumerate(comunidades_usr):
        print(f" - Bolha {i}: {len(comm)} usuários")
        
    # 7.2. Comunidades de Entidades (QP4) - Louvain
    # Agrupa aspectos avaliados conjuntamente e de forma semelhante
    G_ent_pos = nx.Graph()
    G_ent_pos.add_nodes_from(G_ent_proj.nodes())
    for u, v, d in G_ent_proj.edges(data=True):
        if d['weight'] > 0: # correlação positiva de avaliação
            G_ent_pos.add_edge(u, v, weight=d['weight'])
            
    comunidades_ent = community.louvain_communities(G_ent_pos, weight='weight', seed=42)
    partition_ent = {}
    for i, comm in enumerate(comunidades_ent):
        for node in comm:
            partition_ent[node] = i
            
    print(f"\nTotal de {len(comunidades_ent)} grupos de entidades correlacionadas identificados.")
    for i, comm in enumerate(comunidades_ent):
        print(f" - Grupo {i}: {list(comm)}")
        
    return partition_usr, partition_ent

partition_usr, partition_ent = detectar_comunidades_projeções(G_usuarios, G_entidades_proj)


In [ ]:
# 8. Visualizações Gráficas das Redes do Evento
def renderizar_redes_evento(G_bip, G_user, G_ent_proj, partition_usr, partition_ent):
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    
    # 8.1. Plot 1 - Rede Bipartida
    ax1 = axes[0]
    ax1.set_title("Rede Bipartida Sinalizada\n(Usuários -> Entidades)", fontsize=12, fontweight='bold')
    usuarios = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 0]
    entidades = [n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 1]
    
    pos = {}
    pos.update((node, (1, index)) for index, node in enumerate(usuarios))
    pos.update((node, (2, index * (len(usuarios)/len(entidades)) + 1)) for index, node in enumerate(entidades))
    
    nx.draw_networkx_nodes(G_bip, pos, nodelist=usuarios, node_color='lightskyblue', 
                           node_shape='o', node_size=150, ax=ax1, edgecolors='black', linewidths=0.5)
    nx.draw_networkx_nodes(G_bip, pos, nodelist=entidades, node_color='lightcoral', 
                           node_shape='s', node_size=600, ax=ax1, edgecolors='black', linewidths=1.0)
    
    edges = G_bip.edges(data=True)
    pos_edges = [(u, v) for u, v, d in edges if d['weight'] > 0.3]
    neg_edges = [(u, v) for u, v, d in edges if d['weight'] < -0.3]
    neu_edges = [(u, v) for u, v, d in edges if -0.3 <= d['weight'] <= 0.3]
    
    nx.draw_networkx_edges(G_bip, pos, edgelist=pos_edges, edge_color='forestgreen', width=1.2, alpha=0.6, ax=ax1)
    nx.draw_networkx_edges(G_bip, pos, edgelist=neg_edges, edge_color='crimson', width=1.2, alpha=0.6, ax=ax1)
    nx.draw_networkx_edges(G_bip, pos, edgelist=neu_edges, edge_color='gray', width=0.8, alpha=0.2, ax=ax1)
    
    ent_labels = {n: n for n in entidades}
    nx.draw_networkx_labels(G_bip, pos, ent_labels, font_size=8, font_weight='bold', ax=ax1, horizontalalignment='left')
    ax1.axis('off')
    
    # 8.2. Plot 2 - Projeção de Usuários
    ax2 = axes[1]
    ax2.set_title("Projeção de Usuários\n(Bolhas de Concordância)", fontsize=12, fontweight='bold')
    pos_user = nx.spring_layout(G_user, k=0.4, seed=42)
    
    usr_comms = list(set(partition_usr.values()))
    palette_usr = sns.color_palette("Set2", len(usr_comms))
    colors_usr = [palette_usr[partition_usr[node]] for node in G_user.nodes()]
    
    nx.draw_networkx_nodes(G_user, pos_user, node_color=colors_usr, node_size=200, 
                           edgecolors='black', linewidths=0.5, ax=ax2)
    
    user_edges = G_user.edges(data=True)
    pos_u = [(u, v) for u, v, d in user_edges if d['weight'] > 0]
    neg_u = [(u, v) for u, v, d in user_edges if d['weight'] < 0]
    
    nx.draw_networkx_edges(G_user, pos_user, edgelist=pos_u, edge_color='royalblue', width=0.8, alpha=0.3, ax=ax2)
    nx.draw_networkx_edges(G_user, pos_user, edgelist=neg_u, edge_color='darkorange', width=1.0, alpha=0.2, ax=ax2)
    ax2.axis('off')
    
    # 8.3. Plot 3 - Projeção de Entidades
    ax3 = axes[2]
    ax3.set_title("Projeção de Entidades\n(Associação de Aspectos)", fontsize=12, fontweight='bold')
    pos_ent = nx.circular_layout(G_ent_proj)
    
    ent_comms = list(set(partition_ent.values()))
    palette_ent = sns.color_palette("Pastel1", len(ent_comms))
    colors_ent = [palette_ent[partition_ent[node]] for node in G_ent_proj.nodes()]
    
    nx.draw_networkx_nodes(G_ent_proj, pos_ent, node_color=colors_ent, node_size=800, 
                           edgecolors='black', linewidths=1.0, ax=ax3)
    
    ent_edges = G_ent_proj.edges(data=True)
    pos_e = [(u, v) for u, v, d in ent_edges if d['weight'] > 0]
    neg_e = [(u, v) for u, v, d in ent_edges if d['weight'] < 0]
    
    nx.draw_networkx_edges(G_ent_proj, pos_ent, edgelist=pos_e, edge_color='green', width=1.5, alpha=0.5, ax=ax3)
    nx.draw_networkx_edges(G_ent_proj, pos_ent, edgelist=neg_e, edge_color='red', width=1.5, alpha=0.5, ax=ax3)
    
    nx.draw_networkx_labels(G_ent_proj, pos_ent, font_size=9, font_weight='bold', ax=ax3)
    ax3.axis('off')
    
    plt.tight_layout()
    plt.show()

renderizar_redes_evento(G_bipartida, G_usuarios, G_entidades_proj, partition_usr, partition_ent)


## 5. Resultados Preliminares e Discussão

A execução do pipeline estruturado com os dados do evento simulado permitiu extrair as seguintes conclusões estruturais respondendo às QPs:

* **Resposta à QP1 (Concentração e Polarização)**: A análise das entidades indicou que aspectos como *"Internet Wi-Fi"* concentraram a insatisfação (saldo e NSS estrutural negativos expressivos), enquanto o *"Palestrante A"* foi amplamente elogiado. O grau de entrada ponderado e o NSS estrutural oferecem uma medida mais refinada e topológica do que a simples média agregada global do evento, apontando focos pontuais de descontentamento.
* **Resposta à QP2 (Heterogeneidade dos Usuários)**: A tabela de atividade de usuários revela que a participação em feedbacks é heterogênea. A maioria dos usuários contribui com poucos comentários (grau de saída 1 ou 2), mas existem usuários mais engajados (grau de saída 4) cujo feedback abrange diversas entidades. Mapear esses super-usuários ajuda os organizadores a discernir se uma crítica generalizada partiu de um amplo consenso ou de um pequeno grupo de participantes repetitivos.
* **Resposta à QP3 (Bolhas de Opinião Compartilhada)**: O algoritmo de Louvain na rede projetada de usuários ($G_{user}$) encontrou bolhas distintas de concordância de opinião. Os usuários dentro da mesma bolha avaliam as entidades de maneira homofílica, permitindo segmentar o público do evento em perfis de satisfação bem delineados.
* **Resposta à QP4 (Correlação de Aspectos)**: A projeção de entidades ($G_{entity}$) revelou a associação entre os aspectos do evento. Grupos de entidades conectadas por pesos altamente positivos indicam que a satisfação (ou insatisfação) com um serviço (ex.: *Organização* e *Coffee Break*) caminha de forma correlacionada entre os participantes. Arestas negativas sinalizam divergência sistemática.
* **Resposta à QP5 (Sentimento Geral do Evento)**: O NSS Global Tradicional de mercado (-14.49%) oferece um resumo pessimista da percepção agregada. No entanto, a análise de rede bipartida sinalizada revela que a insatisfação se concentrou em serviços específicos (Wi-Fi, Coffee Break e Ar Condicionado), enquanto os Palestrantes obtiveram alta aprovação estrutural. O peso médio das arestas (-0.054) complementa essa visão ao traduzir uma neutralidade geral com picos pontuais, provando que a topologia de rede evita decisões baseadas em médias agregadas enganosas.


## 6. Considerações Éticas e Modelos Generativos

No desenvolvimento deste projeto final, a exploração de Modelos Generativos e as diretrizes éticas foram delineadas da seguinte forma:
1. **Calibração de Dados Sintéticos**: Como destacado pela revisão por pares, dados sintéticos podem introduzir vieses. Para evitar o viés de super-otimismo ou super-coesão apontado no artigo de Nonaka & Perry (2026), a simulação de dados sintéticos para testes e desenvolvimento do pipeline foi calibrada diretamente pelas proporções de sentimentos e conexões observadas nos logs de eventos reais do myMobiConf.
2. **Processamento ABSA e Modelos de Linguagem**: A extração automática de entidades e sentimentos apoia-se em modelos abertos baseados em *Transformers* (como *pysentimiento* ou *BERT*). O uso dessas ferramentas garante transparência reprodutiva e segue os padrões de atribuição científica apropriados.
3. **Anonimização de Feedbacks**: A análise de redes bipartidas liga manifestações de um mesmo participante pelo identificador de sessão. Contudo, todos os dados são previamente descaracterizados, garantindo a privacidade dos participantes e impedindo a reidentificação pessoal de suas opiniões sobre o evento.


## 7. Conclusões e Trabalhos Futuros

Este projeto estabeleceu uma abordagem metodológica robusta para modelar dados de feedback de eventos na plataforma *myMobiConf* através de Redes Bipartidas com Sinais. Ao invés do cálculo simplificado de NSS agregado global, as projeções unipartidas sinalizadas permitiram identificar bolhas de opinião e dependência de aspectos do evento.

**Trabalhos Futuros**:
Como extensões desta pesquisa, sugere-se:
1. Testar dinâmicas de contágio de descontentamento no evento simulando a propagação de insatisfações entre usuários na rede projetada baseada em interações físicas capturadas por sensores internos (Oliveira et al., 2024).
2. Validar o pipeline NLP e a robustez estrutural das projeções em dados reais consolidados de múltiplas edições de conferências acadêmicas suportadas pela plataforma myMobiConf.


## 8. Referências Bibliográficas

* **Blondel, V. D., Guillaume, J.-L., Lambiotte, R., & Lefebvre, E. (2008).** *Fast unfolding of communities in large networks.* Journal of Statistical Mechanics: Theory and Experiment, 2008(10), P10008. [https://doi.org/10.1088/1742-5468/2008/10/P10008](https://doi.org/10.1088/1742-5468/2008/10/P10008)

* **Loughran, T., & McDonald, B. (2011).** *When is a liability not a liability? Textual analysis, dictionaries, and liquidity.* The Journal of Finance, 66(1), 35-65. [https://doi.org/10.1111/j.1540-6261.2010.01625.x](https://doi.org/10.1111/j.1540-6261.2010.01625.x)

* **Newman, M. E. J. (2018).** *Networks.* Oxford University Press. [https://doi.org/10.1093/oso/9780198805090.001.0001](https://doi.org/10.1093/oso/9780198805090.001.0001)

* **Nonaka, H., & Perry, K. E. (2026).** *Evaluating LLM Story Generation through Large-scale Network Analysis of Social Structures.* (Artigo Base, localizado na pasta `artigos/` do projeto).

* **Oliveira, P. G., Braga, T. R. M., & Silva, F. A. (2024).** *Uma Solução de Localização e Navegação Interna para o Sistema myMobiConf.* In: Anais do XVIII Simpósio Brasileiro de Computação Ubíqua e Pervasiva (SBCUP 2024). Porto Alegre: SBC. [https://doi.org/10.5753/sbcup.2024.2389](https://doi.org/10.5753/sbcup.2024.2389)

* **Reichheld, F. F. (2003).** *The One Number You Need to Grow.* Harvard Business Review, 81(12), 46-55. [https://hbr.org/2003/12/the-one-number-you-need-to-grow](https://hbr.org/2003/12/the-one-number-you-need-to-grow)

* **Reyes-Mata, A. E., et al. (2024).** *Prioritizing the Net Sentiment Score: A Banking Industry Case Study.* The Anáhuac Journal, 24(1), 84-106. [https://doi.org/10.25009/aj.v24i1.2612](https://doi.org/10.25009/aj.v24i1.2612)
